# 01 · ¿ML? y Datos (pasos 1 y 2)

**Problema:** dado un vuelo programado, ¿llegará con 15 minutos o más de retraso? Se predice **antes del despegue**.

Ruta: **1 ¿ML?** · 2 Datos · 3 Limpieza · 4 Columnas · 5 Modelos · 6 Evaluación

In [7]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATOS = Path("../datos/crudos")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## 1 · ¿Hace falta ML?

| Pregunta | Respuesta |
|---|---|
| ¿Hay un patrón que aprender? |  |
| ¿Hay etiqueta? |  |
| ¿Hay suficientes datos? | SI ( > 8M) |
| ¿Una regla a mano ya basta? |  |
| ¿Cuánto cuesta equivocarse? |  |

**Veredicto:** 

## 2 · Los datos: la ficha

- **Página:** https://www.kaggle.com/datasets/umerhafeezsiddiqui/2021-2026-us-flights
- **Quién publica los datos:** TODO (autor en Kaggle y fuente original)
- **Cómo se descargan:** (kagglehub)
- **Etiqueta:** `retraso15` = el vuelo llegó con ≥ 15 min de retraso. ¿Por qué vale la pena predecirla?

### ¿Qué archivos hay y cómo se llaman las columnas de verdad?

In [9]:
df = pd.read_csv(DATOS / "Flights_2025_2026.csv")
print(df.shape)
df.head(6)

(8454288, 35)


,YEAR,MONTH,DAY_OF_MONTH,FL_DATE,OP_UNIQUE_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DEL15,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,S
0,2026,4,1,04-01-26 00:00,9E,N131EV,4969,15919,XNA,12953,LGA,1150,1258.0,68.0,1.0,24.0,1322.0,1649.0,7.0,1547,1656.0,69.0,1.0,0,0,177.0,178.0,147.0,1147,0.0,0.0,69.0,0.0,0.0,NaN
1,2026,4,1,04-01-26 00:00,9E,N131EV,5348,12953,LGA,13931,ORF,1633,1743.0,70.0,1.0,47.0,1830.0,1938.0,11.0,1803,1949.0,106.0,1.0,0,0,90.0,126.0,68.0,296,2.0,0.0,36.0,0.0,68.0,NaN
2,2026,4,1,04-01-26 00:00,9E,N131EV,5348,13931,ORF,12953,LGA,1843,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2012,NaN,NaN,NaN,1,0,89.0,NaN,NaN,296,NaN,NaN,NaN,NaN,NaN,NaN
3,2026,4,1,04-01-26 00:00,9E,N131EV,5402,11423,DSM,13487,MSP,655,646.0,-9.0,0.0,16.0,702.0,747.0,7.0,814,754.0,-20.0,0.0,0,0,79.0,68.0,45.0,232,NaN,NaN,NaN,NaN,NaN,NaN
4,2026,4,1,04-01-26 00:00,9E,N131EV,5536,13487,MSP,15919,XNA,904,911.0,7.0,0.0,27.0,938.0,1102.0,4.0,1109,1106.0,-3.0,0.0,0,0,125.0,115.0,84.0,596,NaN,NaN,NaN,NaN,NaN,NaN
5,2026,4,1,04-01-26 00:00,9E,N132EV,4699,10208,AGS,10397,ATL,650,639.0,-11.0,0.0,18.0,657.0,727.0,8.0,802,735.0,-27.0,0.0,0,0,72.0,56.0,30.0,143,NaN,NaN,NaN,NaN,NaN,NaN


> Si los nombres de arriba no coinciden con los de `MAPA_COLUMNAS` (en `src/vuelos/config.py`), **corrijan el mapa** antes de seguir.
> El resto del código usa solo los nombres internos.

### Unir los archivos y quedarnos con mayo 2025 – mayo 2026

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8454288 entries, 0 to 8454287
Data columns (total 35 columns):
 #   Column               Dtype  
---  ------               -----  
 0   YEAR                 int64  
 1   MONTH                int64  
 2   DAY_OF_MONTH         int64  
 3   FL_DATE              str    
 4   OP_UNIQUE_CARRIER    str    
 5   TAIL_NUM             str    
 6   OP_CARRIER_FL_NUM    int64  
 7   ORIGIN_AIRPORT_ID    int64  
 8   ORIGIN               str    
 9   DEST_AIRPORT_ID      int64  
 10  DEST                 str    
 11  CRS_DEP_TIME         int64  
 12  DEP_TIME             float64
 13  DEP_DELAY            float64
 14  DEP_DEL15            float64
 15  TAXI_OUT             float64
 16  WHEELS_OFF           float64
 17  WHEELS_ON            float64
 18  TAXI_IN              float64
 19  CRS_ARR_TIME         int64  
 20  ARR_TIME             float64
 21  ARR_DELAY            float64
 22  ARR_DEL15            float64
 23  CANCELLED            int64  
 24  DIVERTED 

In [17]:
(df.isnull().sum() / df.shape[0]).round(3) *100

YEAR                     0.0
MONTH                    0.0
DAY_OF_MONTH             0.0
FL_DATE                  0.0
OP_UNIQUE_CARRIER        0.0
TAIL_NUM                 0.3
OP_CARRIER_FL_NUM        0.0
ORIGIN_AIRPORT_ID        0.0
ORIGIN                   0.0
DEST_AIRPORT_ID          0.0
DEST                     0.0
CRS_DEP_TIME             0.0
DEP_TIME                 1.7
DEP_DELAY                1.7
DEP_DEL15                1.7
TAXI_OUT                 1.8
WHEELS_OFF               1.8
WHEELS_ON                1.8
TAXI_IN                  1.8
CRS_ARR_TIME             0.0
ARR_TIME                 1.8
ARR_DELAY                2.1
ARR_DEL15                2.1
CANCELLED                0.0
DIVERTED                 0.0
CRS_ELAPSED_TIME         0.0
ACTUAL_ELAPSED_TIME      2.1
AIR_TIME                 2.1
DISTANCE                 0.0
CARRIER_DELAY           77.9
WEATHER_DELAY           77.9
NAS_DELAY               77.9
SECURITY_DELAY          77.9
LATE_AIRCRAFT_DELAY     77.9
S             

### Eliminamos las variables con más del 50% de los datos nulos

In [ ]:
df.drop(['CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'S'], axis=1, inplace=True)

In [ ]:
df.head(5)

,YEAR,MONTH,DAY_OF_MONTH,FL_DATE,OP_UNIQUE_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DEL15,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE
0,2026,4,1,04-01-26 00:00,9E,N131EV,4969,15919,XNA,12953,LGA,1150,1258.0,68.0,1.0,24.0,1322.0,1649.0,7.0,1547,1656.0,69.0,1.0,0,0,177.0,178.0,147.0,1147
1,2026,4,1,04-01-26 00:00,9E,N131EV,5348,12953,LGA,13931,ORF,1633,1743.0,70.0,1.0,47.0,1830.0,1938.0,11.0,1803,1949.0,106.0,1.0,0,0,90.0,126.0,68.0,296
2,2026,4,1,04-01-26 00:00,9E,N131EV,5348,13931,ORF,12953,LGA,1843,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2012,NaN,NaN,NaN,1,0,89.0,NaN,NaN,296
3,2026,4,1,04-01-26 00:00,9E,N131EV,5402,11423,DSM,13487,MSP,655,646.0,-9.0,0.0,16.0,702.0,747.0,7.0,814,754.0,-20.0,0.0,0,0,79.0,68.0,45.0,232
4,2026,4,1,04-01-26 00:00,9E,N131EV,5536,13487,MSP,15919,XNA,904,911.0,7.0,0.0,27.0,938.0,1102.0,4.0,1109,1106.0,-3.0,0.0,0,0,125.0,115.0,84.0,596
